In [2]:
import numpy as np
import pandas as pd
from scipy.sparse import coo_matrix, hstack
from scipy.sparse.linalg import lsmr

from src.dataloaders.simple.common import prepare_detailed_results, prepare_engineered_features
from src.datasets.datasets import (
    detailed_regular_season_results,
    detailed_tourney_results,
    overall_elo_delta,
)
from src.utils.constants import Columns
from src.utils.paths import get_data_directory

In [3]:
MAX_REGULAR_SEASON_DAY = 132
TOURNAMENT_LENGTH = 22

In [4]:
df_regular = prepare_detailed_results(detailed_regular_season_results())
df_tourney = prepare_detailed_results(detailed_tourney_results())

df_overall = (
    pd.concat([df_regular, df_tourney]).sort_values(by=["Season", "DayNum"], ascending=True).reset_index(drop=True)
)
elo_delta = prepare_engineered_features(overall_elo_delta())
df_overall = pd.merge(
    df_overall,
    elo_delta,
    on=[Columns.SEASON, Columns.DAY_NUM, Columns.T1_TEAM_ID, Columns.T2_TEAM_ID],
    how="left",
)

In [5]:
df_matchups = df_overall.copy()[
    [Columns.SEASON, Columns.DAY_NUM, Columns.T1_TEAM_ID, Columns.T2_TEAM_ID, Columns.MEN_WOMEN, Columns.TARGET]
]

### Weighted Season stats

TODO: ensure daynum is smaller than target game (daynum the eventual entry shows to) and only keep teams that actually play on target daynum

In [6]:
def apply_time_discount(games: pd.DataFrame, discount_factor: float = 0.99) -> pd.DataFrame:
    """
    Apply exponential time-based discounting to game weights.

    More recent games (closer to end of season) receive higher weights.

    Args:
        df (pd.DataFrame): DataFrame with games including Season, DayNum, and GameWeight.

    Returns:
        pd.DataFrame: DataFrame with updated GameWeight including time discount.
    """
    INTER_DAY_NUM = "InterDayNum"
    DAYS_FROM_END = "DaysFromEnd"

    df = games.copy()

    df[Columns.DAY_NUM] = df[Columns.DAY_NUM] - df[Columns.DAY_NUM].min()
    df[INTER_DAY_NUM] = df[Columns.DAY_NUM] + 40 * (df[Columns.SEASON] - df[Columns.SEASON].min())

    df[DAYS_FROM_END] = df[INTER_DAY_NUM].max() - df[INTER_DAY_NUM]
    df[Columns.GAME_WEIGHT] = discount_factor ** df[DAYS_FROM_END]

    return df.drop(columns=[INTER_DAY_NUM, DAYS_FROM_END])

In [7]:
def compute_weighted_season_stats(games: pd.DataFrame) -> pd.DataFrame:
    """
    Compute weighted average statistics for each team per season.

    Uses GameWeight column to compute weighted averages across all stat columns.
    Formula: weighted_avg = sum(stat * weight) / sum(weight)

    Args:
        df (pd.DataFrame): DataFrame with game data including GameWeight.

    Returns:
        pd.DataFrame: Aggregated team statistics per season.
    """
    exclude_cols = {
        Columns.SEASON,
        Columns.DAY_NUM,
        Columns.T1_TEAM_ID,
        Columns.T2_TEAM_ID,
        Columns.NUM_OT,
        Columns.MEN_WOMEN,
        Columns.TARGET,
    }
    stat_cols = [col for col in games.columns if col not in exclude_cols]

    games_weighted = games.copy()

    for col in stat_cols:
        if pd.api.types.is_numeric_dtype(games_weighted[col]):
            games_weighted[col] = games_weighted[col] * games_weighted[Columns.GAME_WEIGHT]

    weighted_sums = games_weighted.groupby([Columns.T1_TEAM_ID])[stat_cols].sum()

    weight_sums = games.groupby([Columns.T1_TEAM_ID])[Columns.GAME_WEIGHT].sum()

    weighted_averages = weighted_sums.div(weight_sums, axis=0)

    weighted_averages = weighted_averages.reset_index()
    weighted_averages.drop([Columns.GAME_WEIGHT], inplace=True, axis=1)
    return weighted_averages.rename(columns={Columns.T1_TEAM_ID: Columns.TEAM_ID})

### Team Quality

In [8]:
def compute_quality_glm_antisym(df: pd.DataFrame) -> pd.DataFrame:
    # filter out invalid rows
    df = df[[Columns.T1_TEAM_ID, Columns.T2_TEAM_ID, Columns.POINT_DIFF]].dropna()
    if df.empty:
        return pd.DataFrame(columns=[Columns.TEAM_ID, Columns.QUALITY])

    # treat team IDs as strings to ensure deterministic ordering
    t1 = df[Columns.T1_TEAM_ID].astype(str)
    t2 = df[Columns.T2_TEAM_ID].astype(str)
    teams = np.unique(np.concatenate([t1.values, t2.values]))
    m = len(teams)
    team_to_idx = {team: i for i, team in enumerate(teams)}
    n = len(df)
    rows = np.arange(n)

    # build one‑hot matrices for T1 and T2
    c1 = np.array([team_to_idx[x] for x in t1])
    c2 = np.array([team_to_idx[x] for x in t2])
    X_T1 = coo_matrix((np.ones(n), (rows, c1)), shape=(n, m))
    X_T2 = coo_matrix((np.ones(n), (rows, c2)), shape=(n, m))

    # stacked design X = [X_T1 | X_T2]
    X = hstack([X_T1, X_T2]).tocsr()
    y = df[Columns.POINT_DIFF].to_numpy()

    # minimum‑norm solution to Xβ = y
    beta = lsmr(X, y)[0]
    q, r = beta[:m], beta[m:]

    # identify the baseline team (statsmodels uses the lexicographically
    # smallest team as the omitted category for T2)
    baseline = min(teams)
    shift = r[team_to_idx[baseline]]

    # adjust the T1 block by the baseline T2 coefficient
    q_adjusted = q + shift

    out = pd.DataFrame({Columns.TEAM_ID: [int(t) for t in teams], Columns.QUALITY: q_adjusted})
    return out.sort_values(Columns.TEAM_ID).reset_index(drop=True)

### Aggregate sliding window data

In [9]:
def generate_perspectives(df_season_stats: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    df_season_stats_T1 = df_season_stats.copy()
    df_season_stats_T1.columns = [
        "T1_avg_" + x.replace("T1_", "").replace("T2_", "opponent_") for x in list(df_season_stats_T1.columns)
    ]
    df_season_stats_T1 = df_season_stats_T1.rename(
        {
            "T1_avg_Season": Columns.SEASON,
            "T1_avg_TeamID": Columns.T1_TEAM_ID,
            "T1_avg_LastElo": Columns.T1_ELO,
            "T1_avg_Quality": Columns.T1_QUALITY,
            "T1_avg_Seed": Columns.T1_SEED,
        },
        axis=1,
    )

    df_season_stats_T2 = df_season_stats.copy()
    df_season_stats_T2.columns = [
        "T2_avg_" + x.replace("T1_", "").replace("T2_", "opponent_") for x in list(df_season_stats_T2.columns)
    ]
    df_season_stats_T2 = df_season_stats_T2.rename(
        {
            "T2_avg_Season": Columns.SEASON,
            "T2_avg_TeamID": Columns.T2_TEAM_ID,
            "T2_avg_LastElo": Columns.T2_ELO,
            "T2_avg_Quality": Columns.T2_QUALITY,
            "T2_avg_Seed": Columns.T2_SEED,
        },
        axis=1,
    )
    return df_season_stats_T1, df_season_stats_T2

In [10]:
def add_team_data_for_matchups(matchups_df: pd.DataFrame) -> pd.DataFrame:
    def _prep_side(df_side: pd.DataFrame, key_col: str, prefix: str) -> pd.DataFrame:
        """Use key_col as index, drop it from columns, and ensure prefixed columns."""
        df_side = df_side.copy()
        if key_col in df_side.columns:
            df_side = df_side.set_index(key_col)
            # key is now in the index; make sure it's not in columns anymore
            df_side = df_side.drop(columns=[key_col], errors="ignore")
        # make sure there are no unprefixed columns that could collide
        if not all(c.startswith(prefix) for c in df_side.columns):
            df_side = df_side.add_prefix(prefix)
        # index name doesn't need to match the left for join(on=...)
        return df_side

    # Create a copy to avoid modifying original
    result = matchups_df.copy()
    updated_chunks = []

    # Iterate over each unique Season/DayNum combination
    for (season, day_num), group in matchups_df.groupby([Columns.SEASON, Columns.DAY_NUM]):
        if season == 2003 and day_num < MAX_REGULAR_SEASON_DAY:
            continue

        data = df_overall[
            ((df_overall[Columns.SEASON] == season - 1) & (df_overall[Columns.DAY_NUM] > day_num + TOURNAMENT_LENGTH))
            | ((df_overall[Columns.SEASON] == season) & (df_overall[Columns.DAY_NUM] < day_num))
        ].copy()

        if len(data) == 0:
            continue

        # Calculate stats for this period
        stats = compute_weighted_season_stats(apply_time_discount(data, 0.98))
        team_qualities = compute_quality_glm_antisym(data)
        stats = pd.merge(stats, team_qualities, how="left", on=Columns.TEAM_ID)

        df_T1_stats, df_T2_stats = generate_perspectives(stats)
        df_T1_stats = _prep_side(df_T1_stats, key_col=Columns.T1_TEAM_ID, prefix="T1_")
        df_T2_stats = _prep_side(df_T2_stats, key_col=Columns.T2_TEAM_ID, prefix="T2_")

        updated = group.join(df_T1_stats, on=Columns.T1_TEAM_ID, how="left").join(
            df_T2_stats, on=Columns.T2_TEAM_ID, how="left"
        )

        updated_chunks.append(updated)

    # Combine all updated rows
    # If no groups produced updates, return the input unchanged
    if not updated_chunks:
        return result

    # Concatenate while keeping original indices, then write only new columns back
    updated_all = pd.concat(updated_chunks, axis=0, ignore_index=False)

    # Figure out which columns were added by our merges
    new_cols = [c for c in updated_all.columns if c not in result.columns]
    for c in new_cols:
        result[c] = np.nan

    # Update only rows we touched, preserving overall order
    result.loc[updated_all.index, new_cols] = updated_all[new_cols]

    return result

In [11]:
df = add_team_data_for_matchups(df_matchups)

In [12]:
df = pd.merge(
    df,
    elo_delta,
    on=[Columns.SEASON, Columns.DAY_NUM, Columns.T1_TEAM_ID, Columns.T2_TEAM_ID],
    how="left",
)

In [13]:
df = df.dropna().copy()

In [14]:
df[Columns.POINT_DIFF] = df["T2_avg_Score"] - df["T1_avg_Score"]
df[Columns.QUALITY_DIFF] = df[Columns.T2_QUALITY] - df[Columns.T1_QUALITY]

In [18]:
columns = list(df.columns)
columns

['Season',
 'DayNum',
 'T1_TeamID',
 'T2_TeamID',
 'MenWomen',
 'Target',
 'T1_avg_opponent_OR',
 'T1_avg_opponent_Blk',
 'T1_avg_FTM',
 'T1_avg_FGA',
 'T1_avg_Score',
 'T1_avg_opponent_TO',
 'T1_avg_opponent_DR',
 'T1_avg_TO',
 'T1_avg_opponent_Ast',
 'T1_avg_FGM3',
 'T1_avg_opponent_FGA3',
 'T1_avg_opponent_Stl',
 'T1_avg_FTA',
 'T1_avg_opponent_PF',
 'T1_avg_opponent_FGA',
 'T1_avg_opponent_FTM',
 'T1_avg_FGM',
 'T1_avg_Ast',
 'T1_avg_Blk',
 'T1_avg_PF',
 'T1_avg_Stl',
 'T1_avg_DR',
 'T1_avg_OR',
 'T1_avg_opponent_FGM3',
 'T1_avg_opponent_FTA',
 'T1_avg_opponent_Score',
 'T1_avg_opponent_FGM',
 'T1_avg_FGA3',
 'T1_avg_PointDiff',
 'T1_avg_opponent_Elo',
 'T1_avg_Elo',
 'T1_avg_EloDelta',
 'T1_avg_opponent_EloDelta',
 'T1_avg_EloDiff',
 'T1_avg_EloDeltaDiff',
 'T1_Quality',
 'T2_avg_opponent_OR',
 'T2_avg_opponent_Blk',
 'T2_avg_FTM',
 'T2_avg_FGA',
 'T2_avg_Score',
 'T2_avg_opponent_TO',
 'T2_avg_opponent_DR',
 'T2_avg_TO',
 'T2_avg_opponent_Ast',
 'T2_avg_FGM3',
 'T2_avg_opponent_F

In [42]:
df.to_csv(get_data_directory() / "SlidingWindowAverageData.csv", index=False)

## Test Sliding Window Avg Dataloader

In [1]:
import sys

sys.path.append("..")

from src.dataloaders import SlidingWindowAvgDataLoader
from src.submissions.matchups import generate_matchups

C:\git\code\.venv\Lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
C:\git\code\.venv\Lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statemen

In [3]:
print("=" * 80)
print("Testing SlidingWindowAvgDataLoader")
print("=" * 80)

# Initialize with default parameters
print("\n1. Initializing dataloader...")
dataloader = SlidingWindowAvgDataLoader(
    num_features=100,
    train_split=0.75,
    valid_split=0.25,
    random_seed=42,
)
print("   ✓ Dataloader initialized")

# Setup (this loads and processes all data)
print("\n2. Setting up dataloader (this may take a minute)...")
dataloader.setup()
print("   ✓ Setup complete")

# Get train data
print("\n3. Loading train data...")
X_train, y_train = dataloader.train_data(None, None)
print(f"   ✓ Train data shape: {X_train.shape}")
print(f"   ✓ Train target shape: {y_train.shape}")
print(f"   ✓ Train target distribution: {y_train.value_counts().to_dict()}")

# Get validation data
print("\n4. Loading validation data...")
X_valid, y_valid = dataloader.valid_data()
print(f"   ✓ Valid data shape: {X_valid.shape}")
print(f"   ✓ Valid target shape: {y_valid.shape}")
print(f"   ✓ Valid target distribution: {y_valid.value_counts().to_dict()}")

# Get test data
print("\n4. Loading test data...")
df_matchups = generate_matchups(2025)
X_test = dataloader.test_data(df_matchups)
print(f"   ✓ Valid data shape: {X_test.shape}")

# Print split statistics
print("\n6. Split Statistics:")
total_samples = len(X_train) + len(X_valid)
print(f"   Total samples: {total_samples:,}")
print(f"   Train: {len(X_train):,} ({len(X_train) / total_samples * 100:.1f}%)")
print(f"   Valid: {len(X_valid):,} ({len(X_valid) / total_samples * 100:.1f}%)")

# Print feature information
print("\n7. Feature Information:")
print(f"   Number of features: {len(X_train.columns)}")
print(f"   First 10 features: {list(X_train.columns[:10])}")

# Check for missing values
print("\n8. Data Quality Checks:")
train_missing = X_train.isnull().sum().sum()
valid_missing = X_valid.isnull().sum().sum()
print(f"   Train missing values: {train_missing}")
print(f"   Valid missing values: {valid_missing}")

if train_missing > 0:
    print("\n   Features with missing values in train:")
    missing_features = X_train.isnull().sum()
    missing_features = missing_features[missing_features > 0]
    for feat, count in missing_features.items():
        print(f"     - {feat}: {count} ({count / len(X_train) * 100:.2f}%)")

Testing SlidingWindowAvgDataLoader

1. Initializing dataloader...
   ✓ Dataloader initialized

2. Setting up dataloader (this may take a minute)...
   ✓ Setup complete

3. Loading train data...
   ✓ Train data shape: (296938, 81)
   ✓ Train target shape: (296938,)
   ✓ Train target distribution: {0: 148469, 1: 148469}

4. Loading validation data...
   ✓ Valid data shape: (98980, 81)
   ✓ Valid target shape: (98980,)
   ✓ Valid target distribution: {0: 49490, 1: 49490}

4. Loading test data...
   ✓ Valid data shape: (131407, 81)

6. Split Statistics:
   Total samples: 395,918
   Train: 296,938 (75.0%)
   Valid: 98,980 (25.0%)

7. Feature Information:
   Number of features: 81
   First 10 features: ['QualityDiff', 'EloDiff', 'T2_avg_EloDiff', 'T1_avg_EloDiff', 'EloDeltaDiff', 'T2_avg_EloDelta', 'T1_avg_EloDelta', 'T1_Elo', 'T1_avg_PointDiff', 'T2_Elo']

8. Data Quality Checks:
   Train missing values: 0
   Valid missing values: 0
